# Demo of direct transcription
Variables:
* time step $k$, container index $i$, time interval $\Delta t$
* capacity $C_i$, stored energy $E_i$
* charging and discharging power setpoints $P_{c,i}, P_{d,i}$

Parameters:
* degradation rate $\alpha$ in MWh capacity lost per MWh throughput
* marginal cost of degradation $\beta$ in USD per MWh capacity lost
* real-time price of electricity $c$ in USD

State vector: $\vec{x} = [E_1(0), E_2(0), C_1(0), C_2(0), \ldots, E_1(N), E_2(N), C_1(N), C_2(N)]^\top$

Decision vector: $\vec{u} = [P_{c1}(0), P_{d1}(0), P_{c2}(0), P_{d2}(0), \ldots, P_{c1}(N-1), P_{d1}(N-1), P_{c2}(N-1), P_{d2}(N-1)]^\top$

Objective function: $\min_{\vec{x},\vec{u}} \sum_{k=0}^{N} \sum_{i=1}^2 \underbrace{\alpha \beta \Big(P_{c,i}(k) + P_{d,i}(k)\Big) \Delta t}_\text{aging} + \underbrace{c(k) \Big(P_{c,i}(k)-P_{d,i}(k)\Big) \Delta t}_\text{arbitrage}$

Subject to:

Initial conditions
* Starting capacity: $C_i(0) = C_\text{brand new} \cdot \text{SoH}_\text{start of day}$
* Starting stored energy: $E_i(0) = C_i(0) \cdot \text{SoC}_\text{start of day}$ 

Dynamics
* Energy: $E_i(k+1) = E_i(k) + (P_{c,i}(k) - P_{d,i}(k)) \cdot \Delta t$
* Capacity: $C_i(k+1) = C_i(k) - \alpha (P_{c,i}(k) + P_{d,i}(k)) \cdot \Delta t$

Boundary conditions
* Stored energy must be less than capacity: $E_i(k) \le C_i(k)$
* Charging power setpoint can't exceed hard cap: $P_{c,i}(k) \le P_\text{max}$
* Discharging power setpoint can't exceed hard cap: $P_{d,i}(k) \le P_\text{max}$

Terminal conditions
* Ending stored energy: $E_i(N) \le C_i(N) \cdot \text{SoC}_\text{min}$

### Import packages

In [ ]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt

### Define parameters

In [ ]:
N = 24 # time steps
N_CONTAINERS = 2
DT = 1.0 # hr
ALPHA = 2e-5 # MWh cap lost per MWh throughput
BETA = 200000 # $/MWh

SOC_0 = np.array([0.8, 0.4])
SOH_0 = np.array([0.98, 0.82])
SOC_MIN = 0.3
C_NEW = 2.5 # MWh, new container capacity
P_MAX = 1.0 # MW, hard limit

K = np.array(range(N+1))

In [ ]:

PRICE_MIN = 30 # $ / MWh
PRICE_MAX = 250 # $ / MWh
T_SHIFT = 3 # hr

A = (PRICE_MAX - PRICE_MIN) / 2
MU = (PRICE_MAX + PRICE_MIN) / 2
OMEGA = 2 * np.pi / N

def get_price_forecast(k):
	return -1 * A * np.cos(OMEGA * (k - T_SHIFT)) + MU + np.random.normal(loc=0, scale=4)

price_forecast = np.array([np.round(get_price_forecast(k),2) for k in range(N)])

### Define state & decision variables

In [ ]:
E = cp.Variable((N + 1, N_CONTAINERS), nonneg=True)
C = cp.Variable((N + 1, N_CONTAINERS), nonneg=True)

P_c = cp.Variable((N, N_CONTAINERS), nonneg=True)
P_d = cp.Variable((N, N_CONTAINERS), nonneg=True)

### Define constraints

In [ ]:
constraints = []

# Initial conditions
constraints += [C[0, :] == C_NEW * SOH_0] # starting capacity
constraints += [E[0, :] == C_NEW * np.multiply(SOH_0, SOC_0)] # starting stored energy

# Dynamics
# E[k+1] = E[k] + (P_ch - P_dis) * dt
constraints += [E[1:, :] == E[:-1, :] + (P_c - P_d) * DT]
# C[k+1] = C[k] - alpha * (P_c + P_d) * dt
constraints += [C[1:, :] == C[:-1, :] - ALPHA * (P_c + P_d) * DT]

# Boundary conditions
constraints += [E <= C]				# stored energy can't exceed capacity
constraints += [P_c <= P_MAX]		# power setpoint can't exceed hard cap
constraints += [P_d <= P_MAX]		# power setpoint can't exceed hard cap

# Terminal conditions
constraints += [E[N, :] >= SOC_MIN * C[N, :]] # Terminal E_min

### Define objective function

In [ ]:
arbitrage_cost = cp.sum(cp.multiply(price_forecast[:, None], P_c - P_d) * DT)
aging_cost = cp.sum(BETA * ALPHA * (P_c + P_d) * DT)
objective = cp.Minimize(arbitrage_cost + aging_cost)

### Solve

In [ ]:
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.OSQP)

In [ ]:
fig, axes = plt.subplots(3, figsize=(8,12))

color1 = 'tab:blue'
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Stored Energy (MWh)', color=color1, fontweight='bold')
axes[0].plot(K, E.value[:,0], color=color1, linewidth=2, label='Container 1', marker='o')
axes[0].plot(K, E.value[:,1], color='red', linewidth=2, label='Container 2', marker='o')
axes[0].tick_params(axis='y', labelcolor=color1)
axes[0].set_ylim(0, 2.6) # Slightly above max capacity
axes[0].set_title('BESS Energy Dispatch vs. Market Price')
axes[0].legend()

ax2 = axes[0].twinx()
color2 = 'tab:green'
ax2.set_ylabel('RTM Price ($/MWh)', fontweight='bold')
ax2.step(K[:-1], price_forecast, color=color2, where='post', alpha=0.6, label='Price')
ax2.tick_params(axis='y')

axes[1].plot(K, C.value[:,0] - C.value[0,0], label="Container 1", color=color1, marker='o')
axes[1].plot(K, C.value[:,1] - C.value[0,1], label="Container 2", color='red', marker='o')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Change in Capacity (MWh)', fontweight='bold')
axes[1].set_title("Change in Container Capacity")
axes[1].legend()


axes[2].plot(K[:-1], P_c.value[:,0], marker='o', color='blue', label="C1 Charging")
axes[2].plot(K[:-1], -P_d.value[:,0], marker='o', color='blue', linestyle='--', label="C1 Charging")
axes[2].plot(K[:-1], P_c.value[:,1], marker='o', color='red', label="C2 Discharging")
axes[2].plot(K[:-1], -P_d.value[:,1], marker='o', color='red', linestyle='--', label="C2 Discharging")
axes[2].set_xlabel("Hour of Day")
axes[2].set_ylabel("Power setpoint (MW)")
axes[2].legend()
axes[2].set_title("Power setpoints")

plt.grid(axis='x', linestyle='--')
plt.tight_layout()
plt.show()
